# MedScope AI — Unified Training & Inference

This notebook trains the YOLO and Swin models, saves them, and then boots up the full system (Backend + Frontend) and exposes it via localtunnel.

In [ ]:
#@title 1. Runtime and configuration
import os, sys, json, platform, subprocess, glob, shutil, hashlib
from pathlib import Path

REPO_URL = "https://github.com/prey801/finalyearproject.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
PROJECT_DIR = "/content/finalyearproject" #@param {type:"string"}
YOLO_DATA = "data/raw/roboflow_malaria/data.yaml" #@param {type:"string"}
CLASSIFICATION_DATA = "data" #@param {type:"string"}
YOLO_EPOCHS = 100 #@param {type:"integer"}
YOLO_BATCH = 16 #@param {type:"integer"}
CLASS_EPOCHS = 20 #@param {type:"integer"}
CLASS_BATCH = 32 #@param {type:"integer"}
USE_DRIVE = True #@param {type:"boolean"}
DRIVE_OUTPUT = "/content/drive/MyDrive/medscope-ai/weights" #@param {type:"string"}

try:
    import torch
    DEVICE = "0" if torch.cuda.is_available() else "cpu"
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    DEVICE = "cpu"
    print("PyTorch check failed:", e)
print("Training device:", DEVICE)
SUPABASE_URL = "" #@param {type:"string"}
SUPABASE_ANON_KEY = "" #@param {type:"string"}


In [ ]:
#@title 2. Mount Google Drive (recommended for persistent checkpoints)
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    Path(DRIVE_OUTPUT).mkdir(parents=True, exist_ok=True)

In [ ]:
#@title 3. Clone or update the repository
if not Path(PROJECT_DIR).exists():
    !git clone --branch "$BRANCH" "$REPO_URL" "$PROJECT_DIR"
else:
    %cd $PROJECT_DIR
    !git fetch origin
    !git checkout "$BRANCH"
    !git pull --ff-only origin "$BRANCH"
%cd $PROJECT_DIR
!git status --short
!find . -maxdepth 2 -type f | sort | head -200

In [ ]:
#@title 4. Install system and Python dependencies
!apt-get update -qq
!apt-get install -y -qq libgl1
!python -m pip install -U pip setuptools wheel
!python -m pip install "aiobotocore==2.13.0" "boto3==1.34.106" "botocore==1.34.106"
!python -m pip install -r backend/requirements.txt
!python -m pip install -r models/requirements.txt
!python -m pip install pytest qdrant-client ultralytics timm

import torch
print("CUDA after installation:", torch.cuda.is_available())

In [ ]:
#@title 4b. Download Roboflow Malaria Dataset
# Downloads the YOLO-format malaria blood smear dataset from Roboflow.
# Run this BEFORE cell 5 (dataset validation).
#
# Get your free API key: https://app.roboflow.com -> Settings -> API Keys
# WARNING: Do not commit a real API key to git.
# Store it safely as a Colab Secret (lock icon) named 'ROBOFLOW_API_KEY'.

ROBOFLOW_API_KEY = ""  #@param {type:"string"}
ROBOFLOW_WORKSPACE = "brian-musyoki-s-workspace"  #@param {type:"string"}
ROBOFLOW_PROJECT = "iml-malaria-td7v9"  #@param {type:"string"}
ROBOFLOW_VERSION = 1  #@param {type:"integer"}

import pathlib
from roboflow import Roboflow

if not ROBOFLOW_API_KEY:
    try:
        from google.colab import userdata
        ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
        print("Loaded ROBOFLOW_API_KEY from Colab Secrets.")
    except Exception:
        raise ValueError(
            "ROBOFLOW_API_KEY not set. Paste it into this cell or add it "
            "as a Colab Secret named 'ROBOFLOW_API_KEY'."
        )

dest = pathlib.Path(PROJECT_DIR) / "data/raw/roboflow_malaria"
if dest.exists() and dest.is_dir() and not any(dest.iterdir()):
    dest.rmdir()  # remove empty .gitkeep placeholder
dest.mkdir(parents=True, exist_ok=True)

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
dataset = project.version(ROBOFLOW_VERSION).download("yolov8", location=str(dest), overwrite=True)

print(f"Dataset downloaded to: {dest}")
print("Contents:", sorted(f.name for f in dest.iterdir()))

if (dest / "data.yaml").exists():
    print("OK: data.yaml found -- ready for training.")
else:
    raise RuntimeError(
        "data.yaml missing after download. "
        "Check ROBOFLOW_PROJECT/ROBOFLOW_VERSION and ensure yolov8 export exists."
    )


In [ ]:
#@title 5. Validate datasets before training
from pathlib import Path
import yaml

root = Path(PROJECT_DIR)
yolo_yaml = root / YOLO_DATA
class_data = root / CLASSIFICATION_DATA

assert yolo_yaml.exists(), f"Missing YOLO dataset YAML: {yolo_yaml}. Upload/copy the Roboflow dataset first."
with open(yolo_yaml, "r") as f:
    ycfg = yaml.safe_load(f)
print("YOLO config:", ycfg)

assert class_data.exists(), f"Missing classification data directory: {class_data}"
images = [p for p in class_data.rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}]
print(f"Classification images found: {len(images):,}")
if not images:
    raise RuntimeError("No classification images found. Prepare/split the NIH malaria dataset first.")

In [ ]:
#@title 6. Optional NIH dataset split
# Run this only when data/raw/nih_malaria contains the raw images and processed splits do not exist.
# !python -m scripts.split_data

In [ ]:
#@title 7. Train YOLOv11 detector
%cd $PROJECT_DIR

import os
# Use local SQLite for MLflow -- no server needed in Colab.
os.environ["MLFLOW_TRACKING_URI"] = "sqlite:///mlflow.db"
# Disable Ultralytics built-in MLflow callback to avoid duplicate logging.
os.environ["ULTRALYTICS_MLFLOW"] = "false"

!python -m scripts.train_yolo \\
  --data "$YOLO_DATA" \\
  --epochs "$YOLO_EPOCHS" \\
  --batch "$YOLO_BATCH" \\
  --device "$DEVICE"


In [ ]:
#@title 8. Train Swin Transformer classifier
%cd $PROJECT_DIR
!python -m scripts.train_classification   --data_dir "$CLASSIFICATION_DATA"   --epochs "$CLASS_EPOCHS"   --batch "$CLASS_BATCH"   --device "$DEVICE"

## Run the Full System

Now that training is complete and weights are saved, let's boot the system.

In [ ]:
#@title Install localtunnel
!npm install -g localtunnel

In [ ]:
#@title Download SAM2 Weights
%cd $PROJECT_DIR
!wget -q -O sam2_hiera_small.pt https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt
print("SAM2 weights downloaded!")

In [ ]:
#@title Install Backend Dependencies
!apt-get update -qq
!apt-get install -y -qq redis-server postgresql libgl1
!python -m pip install -q -U pip setuptools wheel
!python -m pip install -q -r backend/requirements.txt
!python -m pip install -q -r models/requirements.txt
!python -m pip install -q pytest qdrant-client nest_asyncio pyngrok

In [ ]:
#@title Start Infrastructure (Redis, Postgres, Qdrant)
!service redis-server start
!service postgresql start

# Set up PostgreSQL
!sudo -u postgres psql -c "CREATE USER medscope WITH PASSWORD 'password';" || true
!sudo -u postgres psql -c "CREATE DATABASE medscope_db OWNER medscope;" || true

# Set up Qdrant
import subprocess, time
from pathlib import Path
%cd $PROJECT_DIR
if not Path("qdrant").exists():
    !wget -q https://github.com/qdrant/qdrant/releases/download/v1.7.0/qdrant-x86_64-unknown-linux-gnu.tar.gz
    !tar -xzf qdrant-x86_64-unknown-linux-gnu.tar.gz

subprocess.Popen(["./qdrant"])
time.sleep(3)
print("Infrastructure services started.")

In [ ]:
#@title Install Frontend Dependencies
%cd $PROJECT_DIR/frontend
!npm install

In [ ]:
#@title Run the Entire System
%cd $PROJECT_DIR
import os, subprocess, time
from pathlib import Path

# Set environment variables for backend (using the newly trained weights!)
os.environ['YOLO_WEIGHTS_PATH'] = str(Path(PROJECT_DIR) / "models/weights/yolov11_malaria_best.pt")
os.environ['SWIN_WEIGHTS_PATH'] = str(Path(PROJECT_DIR) / "models/weights/swin_malaria_best.pth")
os.environ['SAM2_WEIGHTS_PATH'] = str(Path(PROJECT_DIR) / "sam2_hiera_small.pt")

# 1. Start Backend
print("Starting backend...")
backend_process = subprocess.Popen(["uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(5) # Wait for startup

# Expose backend
backend_lt = subprocess.Popen(["lt", "--port", "8000"], stdout=subprocess.PIPE, text=True)
backend_url_line = backend_lt.stdout.readline().strip()
backend_url = backend_url_line.split(" ")[-1] if backend_url_line else "http://localhost:8000"
print(f"Backend exposed at: {backend_url}")

# 2. Configure Frontend with Backend URL
with open(f"{PROJECT_DIR}/frontend/.env.local", "w") as f:
    f.write(f"NEXT_PUBLIC_API_URL={backend_url}\n")
    if "SUPABASE_URL" in globals() and SUPABASE_URL:
        f.write(f"NEXT_PUBLIC_SUPABASE_URL={SUPABASE_URL}\n")
        f.write(f"NEXT_PUBLIC_SUPABASE_PUBLISHABLE_KEY={SUPABASE_ANON_KEY}\n")
        f.write(f"NEXT_PUBLIC_SUPABASE_ANON_KEY={SUPABASE_ANON_KEY}\n")

# 3. Start Frontend
print("Starting frontend...")
frontend_process = subprocess.Popen(["npm", "run", "dev"], cwd=f"{PROJECT_DIR}/frontend")
time.sleep(5) # Wait for compilation

# Expose frontend
frontend_lt = subprocess.Popen(["lt", "--port", "3000"], stdout=subprocess.PIPE, text=True)
frontend_url_line = frontend_lt.stdout.readline().strip()
frontend_url = frontend_url_line.split(" ")[-1] if frontend_url_line else "http://localhost:3000"

print("\n" + "="*60)
print(f"🚀 SYSTEM IS RUNNING!")
print(f"🌍 Access the Frontend Web App here: {frontend_url}")
print(f"⚙️ Backend API is at: {backend_url}")
print("="*60 + "\n")

# Keep the cell running so the servers stay alive
try:
    frontend_process.wait()
except KeyboardInterrupt:
    print("\nShutting down...")
    backend_process.terminate()
    frontend_process.terminate()
    backend_lt.terminate()
    frontend_lt.terminate()